# ML Model Design Playbook — Lighthouse INTEX Dataset

**Role:** Machine learning engineer  
**Data:** `ml-pipelines/lighthouse_csv_v7/` (CSV exports aligned with resident care, fundraising, and operations)

This notebook is a **practical design brief**: it maps business questions to targets, features, engineering, leakage checks, and metrics—then ranks options by ROI and feasibility so you know **what to build first**.

---

## Available signal (by file)

| Domain | Files (key columns) |
|--------|---------------------|
| **Residents & risk** | `residents.csv` — `initial_risk_level`, `current_risk_level`, `case_status`, demographics, `safehouse_id`, stay length |
| **Incidents** | `incident_reports.csv` — `incident_date`, `incident_type`, `severity`, `resolved`, `follow_up_required` |
| **Interventions** | `intervention_plans.csv` — `plan_category`, `status`, `target_value`, `target_date`, `services_provided` |
| **Health** | `health_wellbeing_records.csv` — scores, vitals, checkup flags |
| **Education / visits** | `education_records.csv`, `home_visitations.csv` |
| **Donors & gifts** | `supporters.csv`, `donations.csv`, `donation_allocations.csv`, `in_kind_donation_items.csv` |
| **Social** | `social_media_posts.csv` — engagement, `referral_post_id` link to `donations`, campaign fields |
| **Safehouse ops** | `safehouses.csv`, `safehouse_monthly_metrics.csv` — occupancy, incidents, education/health aggregates |

Use the code cells below to inspect schemas and row counts before modeling.

In [1]:
# Setup: paths and quick schema inspection (run this first)
from pathlib import Path
import pandas as pd

_csv = Path("ml-pipelines/lighthouse_csv_v7")
DATA_DIR = _csv if _csv.is_dir() else Path("lighthouse_csv_v7")
assert DATA_DIR.is_dir(), f"Create or point DATA_DIR to your CSV folder; missing: {DATA_DIR.resolve()}"

CORE_FILES = [
    "residents.csv",
    "incident_reports.csv",
    "intervention_plans.csv",
    "health_wellbeing_records.csv",
    "education_records.csv",
    "home_visitations.csv",
    "supporters.csv",
    "donations.csv",
    "donation_allocations.csv",
    "social_media_posts.csv",
    "safehouse_monthly_metrics.csv",
    "safehouses.csv",
]

def peek_csv(name: str, nrows: int = 3) -> pd.DataFrame:
    path = DATA_DIR / name
    return pd.read_csv(path, nrows=nrows)

rows = {}
for fn in CORE_FILES:
    p = DATA_DIR / fn
    if not p.exists():
        rows[fn] = None
        continue
    # fast row count without loading full file
    with p.open("r", encoding="utf-8") as f:
        rows[fn] = sum(1 for _ in f) - 1
pd.Series(rows, name="approx_rows").to_frame()

,approx_rows
residents.csv,60
incident_reports.csv,100
intervention_plans.csv,180
health_wellbeing_records.csv,534
education_records.csv,534
home_visitations.csv,1337
supporters.csv,60
donations.csv,420
donation_allocations.csv,521
social_media_posts.csv,812


---

## Task 1 — Four high-value ML models

Below: **target**, **inputs (tables/fields)**, **business impact**. All four are grounded in `ml-pipelines/lighthouse_csv_v7`.

### Model A — Resident risk escalation (multi-class or ordinal)

| Item | Specification |
|------|----------------|
| **Target** | Predict whether `current_risk_level` will **worsen** in the next 30/90 days vs baseline (derived from ordered labels: e.g. Low → Medium → High → Critical), or predict next-period risk **category**. |
| **Features** | `residents`: `safehouse_id`, `length_of_stay`, `case_category`, binary `sub_cat_*`, `initial_risk_level`; rolling aggregates from `incident_reports` (count by `severity`/`incident_type`, days since last incident, unresolved ratio); rolling from `health_wellbeing_records` (mean `general_health_score`, slope); `education_records` (`attendance_rate`, `progress_percent` trends); `home_visitations` (visit frequency, `family_cooperation_level`, `safety_concerns_noted` rate). |
| **Business impact** | Earlier staffing and clinical attention; fewer crises; better documentation for funders. |

### Model B — Intervention plan success (classification)

| Item | Specification |
|------|----------------|
| **Target** | Binary: plan reaches **success** by `target_date` (define rule: e.g. `status` in {Completed, On Track} vs {On Hold, Cancelled}) or **meets numeric target** (`target_value` vs realized outcome from linked domain—may need defining join to `education_records`/`health_*` by resident + window). |
| **Features** | `intervention_plans`: `plan_category`, `services_provided` (NLP or multi-hot), `target_value`, time-to-deadline; resident context from `residents` (risk, case type); concurrent load (# active plans per resident); historical incident rate; social worker workload proxy if derivable. |
| **Business impact** | Prioritize plan types and service bundles that work; reduce failed plans and case conference churn. |

### Model C — Donor churn / lapse (classification)

| Item | Specification |
| **Target** | For each `supporter_id`, predict **no donation in next 6–12 months** (or drop in gift frequency) from behavioral history as of a snapshot date. |
| **Features** | Rollups from `donations`: `amount`, `donation_type`, `is_recurring`, `channel_source`, `campaign_name`, inter-donation interval, trend; `supporters`: `supporter_type`, `acquisition_channel`, `region`, tenure since `first_donation_date`; optional link: social-driven gifts via `referral_post_id` → post engagement features aggregated **before** snapshot. |
| **Business impact** | Targeted retention campaigns; higher LTV; lower acquisition cost. |

### Model D — Social post → donation outcome (regression or classification)

| Item | Specification |
| **Target** | **Regression:** `donation_referrals` or `estimated_donation_value_php` (or realized donation sum linked by `referral_post_id` within attribution window). **Classification:** any donation attributed within 7/14 days. |
| **Features** | `social_media_posts`: `platform`, `post_type`, `media_type`, `has_call_to_action`, `content_topic`, `sentiment_tone`, `caption_length`, `is_boosted`, `boost_budget_php`, time features; engagement rates **known at post time** (for training past posts, use only same-day or lag features if predicting future); `follower_count_at_post`. Join **post → donations** via `donations.referral_post_id`. |
| **Business impact** | Content calendar optimization; budget for boosts; channel mix. |

---

## Task 2 — Feature engineering strategies

1. **Time-aware aggregation:** For every resident/supporter/post, fix an **as-of date** $t$; compute rolling 30/90-day sums, means, and trends **using only rows with timestamps $\le t$**. Store `data_cutoff` in feature tables for reproducibility.
2. **Ordinal risk:** Encode `*_risk_level` with **ordered** categories or learnable ordinal loss; keep a separate **“unknown”** if missing.
3. **Sparse incidents:** Use **hierarchical** features: safehouse-level baseline rate + resident residual; add **recency** (days since last High/Critical incident).
4. **Text fields:** `services_provided`, `caption`/`hashtags` — start with **TF-IDF or small embeddings**; keep a **no-text** baseline model for comparison.
5. **Donor sequences:** Build **RFM-style** features (recency, frequency, monetary) per snapshot; add **seasonality** (month-of-year) and **campaign** one-hot or target encoding with cross-validation inside folds.
6. **Safehouse context:** Join `safehouse_monthly_metrics` for **peer-relative** features (z-score of incident_count vs same month last year) to normalize operational pressure.
7. **Leakage-safe labels for interventions:** Success must use **outcomes observed after** `created_at` + grace period; do not use `updated_at` as a feature if it is set after outcome is known.

---

## Task 3 — Data leakage risks

| Risk | Where it hides | Mitigation |
|------|----------------|------------|
| **Future information in features** | `current_risk_level` includes post-period knowledge; `resolved`, `resolution_date` for incidents in the prediction window | Train labels from **forward windows**; features strictly **before** horizon start. Drop or time-shift outcome columns. |
| **Target encoded in IDs** | `notes`, free-text containing outcome | Exclude or scrub; use structured fields only for v1. |
| **Post-hoc aggregates** | `safehouse_monthly_metrics` for month $M$ may be computed using full-month incidents | For month-level prediction, use **only prior months** or **intra-month partials** with clear definition. |
| **Donation leakage** | `estimated_donation_value_php` on posts may be fitted from same-period donations | Prefer **realized** donations from `donations` joined by `referral_post_id`, or build label from **held-out** transactions not used in features. |
| **Duplicate information** | `amount` and `estimated_value` both in `donations` | Use one canonical monetary field; document currency. |
| **Train/test by resident/supporter overlap** | Random row split leaks person history | **Group split** by `resident_id` / `supporter_id` / `safehouse_id` where appropriate. |

---

## Task 4 — Evaluation metrics

| Problem type | Metrics | Notes |
|--------------|---------|-------|
| Risk escalation (imbalanced) | **PR-AUC**, recall at fixed precision, **calibrated** probabilities | Report by safehouse stratum; cost-weighted recall if critical class is rare. |
| Intervention success | **Brier score**, PR-AUC, calibration | Balance precision (avoid wasted resources) vs recall (avoid harm). |
| Donor churn | **PR-AUC**, lift at top decile, **expected dollar lift** | Align with CRM campaign capacity (top-k). |
| Social → donation | **MAE / RMSE** on revenue; **CRPS** if probabilistic; ranking: **Spearman** on referral count | Compare to **seasonal naive** and **historical mean by platform**. |

---

## ROI vs feasibility — ranked

| Rank | Model | ROI (relative) | Feasibility | Rationale |
|------|--------|----------------|-------------|-----------|
| 1 | **C — Donor churn** | High | High | Clean tabular signal (`donations` + `supporters`), simple snapshot labels, direct tie to fundraising ROI. |
| 2 | **D — Social → donation** | High | Medium | Strong features in `social_media_posts`; must handle attribution window and leakage carefully. |
| 3 | **A — Risk escalation** | Very high (safety) | Medium | Rich multi-table history; labeling and class imbalance need care; governance for deployment. |
| 4 | **B — Intervention success** | Medium–High | Lower | Outcome definition and join to `target_value` vs actuals is **ambiguous** without extra business rules. |

### Recommendation — build **first**: **Model C (donor churn / lapse)**

- **Why:** Fastest path to a **deployable** model (tabular, clear label, measurable campaign uplift), fewer ambiguous joins than intervention success, less regulatory sensitivity than resident risk scoring.
- **Next:** **Model D** if marketing owns the roadmap; **Model A** when case management partners agree on risk horizons and evaluation ethics.

Optional: run the template below to prototype **churn labels** from `donations.csv`.

In [2]:
# Template: donor churn label (example — adjust horizons to your business definition)
import pandas as pd
from pathlib import Path

_csv = Path("ml-pipelines/lighthouse_csv_v7")
DATA_DIR = _csv if _csv.is_dir() else Path("lighthouse_csv_v7")
don = pd.read_csv(DATA_DIR / "donations.csv", parse_dates=["donation_date"])
sup = pd.read_csv(DATA_DIR / "supporters.csv", parse_dates=["created_at", "first_donation_date"])

OBS_DATE = pd.Timestamp("2024-06-30")  # example observation time
HORIZON_DAYS = 365
LOOKBACK_DAYS = 730

don_hist = don[(don["donation_date"] <= OBS_DATE) & (don["donation_date"] >= OBS_DATE - pd.Timedelta(days=LOOKBACK_DAYS))]
feat = (
    don_hist.groupby("supporter_id")
    .agg(
        n_gifts=("donation_id", "count"),
        total_php=("amount", lambda s: pd.to_numeric(s, errors="coerce").sum()),
        last_gift=("donation_date", "max"),
        first_in_window=("donation_date", "min"),
    )
    .reset_index()
)
feat["recency_days"] = (OBS_DATE - feat["last_gift"]).dt.days

don_future = don[(don["donation_date"] > OBS_DATE) & (don["donation_date"] <= OBS_DATE + pd.Timedelta(days=HORIZON_DAYS))]
future_any = don_future.groupby("supporter_id").size().rename("future_gifts").reset_index()
label = feat.merge(future_any, on="supporter_id", how="left")
label["churned"] = (label["future_gifts"].fillna(0) == 0).astype(int)

label.head(), label["churned"].value_counts()

(   supporter_id  n_gifts  total_php  last_gift first_in_window  recency_days  \
 0             1        7    5458.79 2024-01-08      2023-03-25           174   
 1             2        3    3480.08 2023-09-25      2023-03-08           279   
 2             3        5     710.56 2024-02-06      2023-02-22           145   
 3             4        4    3069.59 2024-04-11      2023-03-15            80   
 4             5        2    2022.07 2024-01-18      2023-12-20           164   
 
    future_gifts  churned  
 0           2.0        0  
 1           1.0        0  
 2          10.0        0  
 3           3.0        0  
 4           2.0        0  ,
 churned
 0    45
 1    10
 Name: count, dtype: int64)

---

## Implementation checklist (before production)

- [ ] Document **prediction time** and **label window** for each model.
- [ ] **Group-based** train/validation/test splits; report **time-based** backtest.
- [ ] Baseline: **logistic regression / gradient boosting** with monotonic constraints on risk if required.
- [ ] **Calibration** for probability outputs; human review for resident-facing scores.
- [ ] **Monitoring:** feature drift, label delay for donations, incident reporting bias.

Extend this notebook with EDA plots and final feature lists as the project matures.